# 02 - Fine-tuning `cahya/wav2vec2-large-xlsr-sundanese`

**IF4071 Fase II - ASR bahasa Sunda dengan wav2vec 2.0 + CTC.**

## A. Pendahuluan
**Fine-tuning** melanjutkan pelatihan terawasi pada checkpoint yang sudah mampu melakukan ASR dengan data berlabel milik kami; ini merupakan bentuk **adaptasi domain** terhadap penutur, kondisi rekaman, dan bahasa Sunda spontan yang bercampur kode. Pelatihan memakai **loss CTC**: model menghasilkan distribusi karakter/*blank* per ~20 ms dan loss menjumlahkan seluruh perataan yang runtuh menjadi teks target sehingga tidak diperlukan penyelarasan per frame.

Ujaran Sunda spontan sulit dikenali karena sumber dayanya terbatas, penuturnya beragam (penutur asli dan non-asli), terdapat kata pengisi dan ketidaklancaran, sebagian rekaman berderau, terjadi alih kode ke bahasa Indonesia/Jawa/Inggris, dan ejaannya informal.

**Eksperimen ini** melatih model pada penutur **latih** yang tetap, memantau dan memilih checkpoint pada penutur **validasi** (CER validasi terbaik; WER juga dicatat), lalu melaporkan WER/CER validasi. Himpunan uji **tidak pernah dimuat**. 

Checkpoint ini sudah di-fine-tune untuk bahasa Sunda (CTC); kami melanjutkan dari titik tersebut.

## B. Pengaturan lingkungan
Prasyarat: **GPU T4** dan **Internet = On** untuk unduhan awal checkpoint (atau lampirkan checkpoint sebagai dataset Kaggle dan arahkan `MODEL_ID` ke path-nya). Bila paket baru terpasang lalu impor gagal, mulai ulang sesi (Run > Restart session) dan jalankan dari atas.

In [ ]:
# Ubah sesuai nama dataset Kaggle yang dilampirkan melalui Add Input.
DATASET_ROOT = "/kaggle/input/<dataset-slug>"
OUTPUT_ROOT = "/kaggle/working/asr_outputs"  # Folder input Kaggle hanya dapat dibaca
SEED = 42
USE_ALL_GPUS = True  # Isi False untuk memakai satu GPU saja bila pelatihan multi-GPU bermasalah

import importlib.util
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
import inspect
import json
import os
import platform
import random
import re
import sys
import unicodedata

# Harus diatur sebelum torch memakai CUDA agar GPU kedua benar-benar tersembunyi.
if not USE_ALL_GPUS:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Kaggle sudah menyertakan torch dan transformers sehingga hanya paket yang belum ada yang dipasang.
missing = [p for p in ("jiwer", "accelerate", "soundfile") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Terpasang:", missing, "- bila impor gagal, pilih Run > Restart session lalu jalankan ulang.")

from huggingface_hub import model_info
from transformers import EarlyStoppingCallback, Trainer
from transformers import TrainingArguments
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2ForCTC, Wav2Vec2Processor
from transformers.trainer_utils import get_last_checkpoint
import hashlib
import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import transformers

if not os.path.isdir(DATASET_ROOT):
    attached = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []
    raise FileNotFoundError(f"DATASET_ROOT {DATASET_ROOT!r} tidak ditemukan. Input terlampir: {attached}. "
                            "Isi dengan folder yang memuat splits/ dan audio/.")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

## Kode pembantu (mandiri)
Seluruh fungsi pembantu (normalisasi teks, metrik, verifikasi pembagian data, utilitas model, dan pipeline pelatihan) didefinisikan pada sel di bawah ini sehingga notebook **tidak memerlukan berkas atau impor tambahan** selain dataset terlampir (`audio/` dan `splits/`). Notebook 02 dan 03 memakai kode pembantu yang sama agar kedua eksperimen fine-tuning sebanding.

In [ ]:
NORMALIZATION_VERSION = "v1"


ALLOWED_CHARS = set("abcdefghijklmnopqrstuvwxyz\u00e9 ")


REMOVED_TAGS = {"ee", "ee..", "noise", "overlap", "s"}


REVIEW_TAGS = {"unclear"}


DIGITS = {"0": "nol", "1": "hiji", "2": "dua", "3": "tilu", "4": "opat",
          "5": "lima", "6": "genep", "7": "tujuh", "8": "dalapan", "9": "salapan"}


# Satu-satunya token gabungan huruf dan angka di korpus sehingga bentuk lisannya ditetapkan secara eksplisit.
SPOKEN_FORMS = {"s2": "es dua"}


_TAG_RE = re.compile(r"\[([^\]]*)\]")


_NUMBER_RE = re.compile(r"(?<![^\W\d_])\d+(?![^\W\d_])")


class NormalizationError(ValueError):
    pass


@dataclass
class NormalizedText:
    text: str
    needs_review: bool
    flags: tuple


def _spell_number(num: str) -> str:
    if len(num) > 2 or (len(num) > 1 and num.startswith("0")):
        raise NormalizationError(f"Angka {num!r} di luar pemetaan eksplisit 0-99.")
    n = int(num)
    if n < 10:
        return DIGITS[num]
    tens, unit = divmod(n, 10)
    if n == 10:
        return "sapuluh"
    if n == 11:
        return "sabelas"
    if tens == 1:
        return f"{DIGITS[str(unit)]} welas"
    return f"{DIGITS[str(tens)]} puluh" + (f" {DIGITS[str(unit)]}" if unit else "")


def normalize_transcript(raw: str) -> NormalizedText:
    if not isinstance(raw, str):
        raise NormalizationError(f"Transkripsi bukan string: {raw!r}")
    flags = []
    text = unicodedata.normalize("NFC", raw).lower()

    def handle_tag(m):
        inner = " ".join(m.group(1).split())
        if inner in REMOVED_TAGS:
            return " "
        if inner in REVIEW_TAGS:
            flags.append("unclear")
            return " "
        flags.append(f"unwrapped:{inner}")
        return f" {inner} "

    text = _TAG_RE.sub(handle_tag, text)
    for written, spoken in SPOKEN_FORMS.items():
        text = re.sub(rf"\b{re.escape(written)}\b", spoken, text)
    text = _NUMBER_RE.sub(lambda m: f" {_spell_number(m.group(0))} ", text)
    if re.search(r"\d", text):
        raise NormalizationError(f"Angka yang menempel pada huruf tidak didukung: {raw!r}")
    text = re.sub(r"[-\u2010-\u2015]", " ", text)
    text = "".join(ch if (ch.isalpha() or ch.isspace()) else " " for ch in text)
    text = " ".join(text.split())
    bad = sorted(set(text) - ALLOWED_CHARS)
    if bad:
        raise NormalizationError(f"Karakter tidak didukung {bad} setelah normalisasi: {raw!r}")
    return NormalizedText(text, "unclear" in flags, tuple(flags))


def normalize_text(raw: str) -> str:
    return normalize_transcript(raw).text


def fold_diacritics(text: str) -> str:
    """Hanya dipakai untuk metrik sekunder tanpa aksen dan tidak pernah dipakai sebagai target latih."""
    return text.replace("\u00e9", "e")


def load_audio(path, expected_sr: int = 16000) -> np.ndarray:
    """Memuat gelombang mono float32 tanpa resampling dan menolak format yang tidak sesuai."""
    wav, sr = sf.read(str(path), dtype="float32", always_2d=True)
    if sr != expected_sr:
        raise ValueError(f"{path}: laju sampel {sr} != {expected_sr}")
    if wav.shape[1] != 1:
        raise ValueError(f"{path}: diharapkan mono, ditemukan {wav.shape[1]} kanal")
    if wav.shape[0] == 0:
        raise ValueError(f"{path}: audio kosong")
    return wav[:, 0]


def pad_ctc_labels(label_ids: list, pad_value: int = -100) -> np.ndarray:
    """Posisi padding bernilai -100 agar diabaikan oleh loss CTC."""
    out = np.full((len(label_ids), max(len(x) for x in label_ids)), pad_value, dtype=np.int64)
    for i, ids in enumerate(label_ids):
        out[i, : len(ids)] = ids
    return out


def _clean(s) -> str:
    return " ".join(str(s).split())


def compute_error_rates(references, predictions, normalize: bool = True) -> dict:
    """Menghitung WER dan CER sebagai total edit dibagi total unit referensi. Prediksi kosong dihitung sebagai penghapusan penuh."""
    refs, hyps = list(references), [_clean(h) for h in predictions]
    if len(refs) != len(hyps):
        raise ValueError(f"{len(refs)} referensi vs {len(hyps)} prediksi")
    if not refs:
        raise ValueError("Tidak ada ujaran untuk dinilai")
    refs = [normalize_text(r) if normalize else _clean(r) for r in refs]
    if any(r == "" for r in refs):
        raise ValueError("Referensi kosong ditemukan; perbaiki manifes")
    wo = jiwer.process_words(refs, hyps)
    co = jiwer.process_characters(refs, hyps)
    return {
        "wer": wo.wer, "cer": co.cer, "n_utterances": len(refs),
        "word_ref_count": wo.hits + wo.substitutions + wo.deletions,
        "char_ref_count": co.hits + co.substitutions + co.deletions,
        "word_sub": wo.substitutions, "word_del": wo.deletions, "word_ins": wo.insertions,
        "char_sub": co.substitutions, "char_del": co.deletions, "char_ins": co.insertions,
    }


def verify_saved_split(split_dir) -> dict:
    """Memastikan CSV tersimpan konsisten dengan metadata dan tidak ada penutur maupun ujaran yang tumpang tindih."""
    d = Path(split_dir)
    meta = json.loads((d / "split_metadata.json").read_text())
    seen = {}
    for name in ("train", "val", "test"):
        if (d / f"{name}.csv").exists():
            df = pd.read_csv(d / f"{name}.csv")
            if set(df["speaker_id"]) != set(meta["subsets"][name]["speakers"]):
                raise AssertionError(f"Penutur {name}.csv berbeda dari split_metadata.json")
            seen[name] = df
    names = list(seen)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            if not set(seen[a]["speaker_id"]).isdisjoint(seen[b]["speaker_id"]):
                raise AssertionError(f"Penutur tumpang tindih antara {a} dan {b}")
            if not set(seen[a]["file"]).isdisjoint(seen[b]["file"]):
                raise AssertionError(f"Ujaran tumpang tindih antara {a} dan {b}")
    return meta


def load_split(dataset_root, name: str, check_audio: bool = True) -> pd.DataFrame:
    """Memuat CSV pembagian dan menambahkan `abs_audio_path` relatif terhadap dataset_root."""
    root = Path(dataset_root)
    df = pd.read_csv(root / "splits" / f"{name}.csv")
    df["abs_audio_path"] = df["audio_path"].map(lambda p: str(root / p))
    if check_audio:
        missing = [p for p in df["abs_audio_path"] if not Path(p).exists()]
        if missing:
            raise FileNotFoundError(f"{len(missing)} berkas audio tidak ditemukan di {root}, mis. {missing[:3]}")
    return df


def env_info() -> dict:
    info = {"python": sys.version.split()[0], "platform": platform.platform(),
            "torch": torch.__version__, "transformers": transformers.__version__,
            "cuda_available": torch.cuda.is_available(), "cuda": torch.version.cuda}
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info.update(gpu=p.name, gpu_mem_gb=round(p.total_memory / 1e9, 1), n_gpus=torch.cuda.device_count())
    return info


def gpu_mem() -> str:
    if not torch.cuda.is_available():
        return "tanpa CUDA"
    return f"terpakai {torch.cuda.memory_allocated() / 1e9:.2f} GB / dicadangkan {torch.cuda.memory_reserved() / 1e9:.2f} GB"


def load_model_and_processor(model_id: str, device: str = "cuda"):
    processor = Wav2Vec2Processor.from_pretrained(model_id)
    return Wav2Vec2ForCTC.from_pretrained(model_id).to(device), processor


def model_revision(model_id: str):
    try:
        return model_info(model_id).sha
    except Exception:  # luring, atau checkpoint dari dataset Kaggle lokal
        return None


def tokenizer_audit(processor, model, texts) -> dict:
    """Melaporkan vocab dan karakter target yang tidak dapat di-encode oleh tokenizer."""
    tok = processor.tokenizer
    vocab = tok.get_vocab()
    needed = set("".join(texts).replace(" ", ""))
    missing = sorted(c for c in needed if c not in vocab)
    return {
        "vocab_size": len(vocab), "model_vocab_size": model.config.vocab_size,
        "vocab": sorted(vocab, key=vocab.get), "word_delimiter": tok.word_delimiter_token,
        "pad_token_id": tok.pad_token_id, "unk_token": tok.unk_token,
        "expected_sampling_rate": processor.feature_extractor.sampling_rate,
        "do_normalize": processor.feature_extractor.do_normalize,
        "missing_target_chars": missing,
        "ctc_loss_reduction": model.config.ctc_loss_reduction,
    }


def add_missing_chars(processor, model, chars, tmp_dir):
    """Menambahkan karakter yang hilang sebagai token CTC baru dan memperbesar lm_head.

    Tokenizer dibangun ulang dari vocab yang diperluas agar token baru benar-benar menjadi
    bagian vocab dan ikut tersimpan. Baris lm_head asli dipertahankan dan hanya baris baru
    yang diinisialisasi.
    """
    old = processor.tokenizer
    vocab = dict(old.encoder)  # Memakai vocab dasar karena get_vocab juga memuat token tambahan bos dan eos
    chars = [c for c in chars if c not in old.get_vocab()]
    if not chars:
        return processor, model
    old_n = model.lm_head.out_features
    if old_n != len(vocab) or max(vocab.values()) != old_n - 1:
        raise RuntimeError(f"lm_head ({old_n}) dan vocab tokenizer ({len(vocab)}) tidak selaras; "
                           "perluasan tidak aman.")
    for i, c in enumerate(chars):
        vocab[c] = old_n + i
    Path(tmp_dir).mkdir(parents=True, exist_ok=True)
    vocab_file = Path(tmp_dir) / "vocab.json"
    vocab_file.write_text(json.dumps(vocab, ensure_ascii=False))
    tok = Wav2Vec2CTCTokenizer(str(vocab_file), unk_token=old.unk_token, pad_token=old.pad_token,
                               bos_token=old.bos_token, eos_token=old.eos_token,
                               word_delimiter_token=old.word_delimiter_token)
    processor = Wav2Vec2Processor(feature_extractor=processor.feature_extractor, tokenizer=tok)
    old_head = model.lm_head
    new_head = torch.nn.Linear(old_head.in_features, len(vocab))
    with torch.no_grad():
        new_head.weight[:old_n] = old_head.weight
        new_head.bias[:old_n] = old_head.bias
        torch.nn.init.normal_(new_head.weight[old_n:], std=0.02)
        torch.nn.init.zeros_(new_head.bias[old_n:])
    model.lm_head = new_head.to(old_head.weight.device)
    model.config.vocab_size = len(vocab)
    return processor, model


def count_unk_targets(processor, texts) -> int:
    """Menghitung target yang masih menjadi token UNK setelah encoding dan nilainya harus nol."""
    unk = processor.tokenizer.unk_token_id
    return sum(unk in processor.tokenizer(t).input_ids for t in texts)


def transcribe_files(model, processor, paths, batch_size: int = 4, device: str = "cuda", fp16: bool = True):
    """Dekode CTC greedy dengan berkas terpanjang lebih dulu agar padding minimal, lalu urutan asli dipulihkan."""
    sr = processor.feature_extractor.sampling_rate
    waves = [load_audio(p, sr) for p in paths]
    order = np.argsort([-len(w) for w in waves])
    preds = [None] * len(waves)
    model.eval()
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        feats = processor.feature_extractor([waves[j] for j in idx], sampling_rate=sr,
                                            padding=True, return_tensors="pt", return_attention_mask=True)
        extra = {"attention_mask": feats["attention_mask"].to(device)} if "attention_mask" in feats else {}
        with torch.no_grad(), torch.autocast("cuda", enabled=fp16 and device == "cuda"):
            logits = model(feats["input_values"].to(device), **extra).logits
        for j, row in zip(idx, torch.argmax(logits, dim=-1).cpu()):
            preds[j] = processor.tokenizer.decode(row, skip_special_tokens=True)
    return [" ".join(p.split()) for p in preds]


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def require_cuda() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU CUDA tidak terdeteksi. Di Kaggle: Settings > Accelerator > GPU T4. "
                           "Fine-tuning penuh di CPU sengaja tidak didukung.")


class CTCDataset:
    """Memuat gelombang 16 kHz saat dibutuhkan dan mengembalikan input ternormalisasi beserta id label."""

    def __init__(self, df: pd.DataFrame, processor):
        self.df, self.processor = df.reset_index(drop=True), processor
        self.sr = processor.feature_extractor.sampling_rate

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        wav = load_audio(row["abs_audio_path"], self.sr)
        return {"input_values": self.processor.feature_extractor(wav, sampling_rate=self.sr).input_values[0],
                "labels": self.processor.tokenizer(row["transcription_normalized"]).input_ids}


@dataclass
class CTCDataCollator:
    """Padding dinamis yang mengisi gelombang dengan nol dan label dengan nilai yang diabaikan CTC."""
    processor: object

    def __call__(self, features):
        batch = self.processor.feature_extractor.pad(
            [{"input_values": f["input_values"]} for f in features], padding=True, return_tensors="pt")
        batch["labels"] = torch.from_numpy(pad_ctc_labels([f["labels"] for f in features], -100))
        return batch


def make_compute_metrics(processor):
    def compute(pred):
        ids = np.argmax(pred.predictions, axis=-1)
        label_ids = np.where(pred.label_ids == -100, processor.tokenizer.pad_token_id, pred.label_ids)
        hyp = processor.batch_decode(ids)
        ref = processor.batch_decode(label_ids, group_tokens=False)
        r = compute_error_rates(ref, hyp, normalize=False)
        return {"wer": r["wer"], "cer": r["cer"]}
    return compute


def build_training_args(output_dir, cfg: dict, **overrides) -> TrainingArguments:
    t = cfg["training"]
    kw = dict(
        output_dir=str(output_dir), learning_rate=float(t["learning_rate"]),
        num_train_epochs=t["num_train_epochs"],
        per_device_train_batch_size=t["per_device_train_batch_size"],
        per_device_eval_batch_size=t["per_device_eval_batch_size"],
        gradient_accumulation_steps=t["gradient_accumulation_steps"],
        weight_decay=t["weight_decay"],
        fp16=t["fp16"], gradient_checkpointing=t["gradient_checkpointing"],
        save_strategy="epoch", save_total_limit=t["save_total_limit"], logging_steps=10,
        load_best_model_at_end=True, metric_for_best_model=t["metric_for_best_model"],
        greater_is_better=t["greater_is_better"], seed=cfg["seed"], data_seed=cfg["seed"],
        report_to="none", remove_unused_columns=False, dataloader_num_workers=2,
    )
    # Nama argumen warmup dan strategi evaluasi berbeda antar versi Transformers
    fields = TrainingArguments.__dataclass_fields__
    kw["warmup_ratio" if "warmup_ratio" in fields else "warmup_steps"] = t["warmup_ratio"]
    kw["eval_strategy" if "eval_strategy" in fields else "evaluation_strategy"] = "epoch"
    return TrainingArguments(**{**kw, **overrides})


def last_checkpoint(output_dir):
    d = Path(output_dir)
    return get_last_checkpoint(str(d)) if d.exists() else None


def history_frame(trainer) -> pd.DataFrame:
    return pd.DataFrame(trainer.state.log_history)


def plot_history(hist: pd.DataFrame):
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    train = hist[hist["loss"].notna()]
    ev = hist[hist["eval_loss"].notna()]
    ax[0].plot(train["epoch"], train["loss"], label="loss latih")
    ax[0].plot(ev["epoch"], ev["eval_loss"], "o-", label="loss validasi")
    ax[1].plot(ev["epoch"], ev["eval_wer"], "o-", color="C1")
    ax[2].plot(ev["epoch"], ev["eval_cer"], "o-", color="C2")
    for a, title in zip(ax, ("Loss", "WER validasi", "CER validasi")):
        a.set_title(title)
        a.set_xlabel("epoch")
        a.grid(alpha=.3)
    ax[0].legend()
    fig.tight_layout()
    return fig


def save_artifacts(trainer, processor, out_dir, cfg_record: dict, val_df, preds, metrics: dict) -> None:
    out_dir = Path(out_dir)
    trainer.save_model(str(out_dir / "model"))
    processor.save_pretrained(str(out_dir / "processor"))
    (out_dir / "training_config.json").write_text(json.dumps(cfg_record, indent=2, ensure_ascii=False, default=str))
    history_frame(trainer).to_csv(out_dir / "training_history.csv", index=False)
    pd.DataFrame({"file": val_df["file"], "speaker_id": val_df["speaker_id"],
                  "reference": val_df["transcription_normalized"], "prediction": preds}
                 ).to_csv(out_dir / "validation_predictions.csv", index=False)
    (out_dir / "validation_metrics.json").write_text(json.dumps(metrics, indent=2))


CFG = {'seed': 42,
 'sampling_rate': 16000,
 'normalization_version': 'v1',
 'training': {'learning_rate': 1e-05,
              'num_train_epochs': 10,
              'per_device_train_batch_size': 1,
              'per_device_eval_batch_size': 1,
              'gradient_accumulation_steps': 8,
              'weight_decay': 0.01,
              'warmup_ratio': 0.1,
              'fp16': True,
              'gradient_checkpointing': True,
              'freeze_feature_encoder': True,
              'save_total_limit': 2,
              'metric_for_best_model': 'eval_cer',
              'greater_is_better': False}}

In [ ]:
MODEL_ID = "cahya/wav2vec2-large-xlsr-sundanese"
RUN_NAME = "cahya_finetuned"
RUN_DIR = os.path.join(OUTPUT_ROOT, RUN_NAME)
OVERWRITE = False                    # Isi True untuk mengizinkan penimpaan hasil akhir di RUN_DIR
RESUME_FROM_CHECKPOINT = True        # Melanjutkan dari checkpoint terakhir bila tersedia
ALLOW_VOCAB_EXTENSION = True    # Penjelasan ada di bagian D

require_cuda()
DEVICE = "cuda"
set_seed(SEED)
print(json.dumps(env_info(), indent=2))
print(torch.cuda.get_device_name(0), "|", gpu_mem())

## C. Pemuatan dataset
Manifes **latih** dan **validasi** yang tersimpan dimuat (pembagian yang sama untuk seluruh notebook).

In [ ]:
meta = verify_saved_split(os.path.join(DATASET_ROOT, "splits"))
# Trainer membagi batch ke semua GPU sehingga akumulasi dibagi jumlah GPU agar batch efektif tetap sama.
n_gpu = max(torch.cuda.device_count(), 1)
base_accum = CFG["training"]["gradient_accumulation_steps"]
assert base_accum % n_gpu == 0, f"akumulasi {base_accum} tidak habis dibagi {n_gpu} GPU"
cfg = {**CFG, "training": {**CFG["training"], "gradient_accumulation_steps": base_accum // n_gpu}}
effective_batch = cfg["training"]["per_device_train_batch_size"] * n_gpu * cfg["training"]["gradient_accumulation_steps"]
print(f"{n_gpu} GPU | akumulasi gradien {cfg['training']['gradient_accumulation_steps']} | batch efektif {effective_batch}")
train = load_split(DATASET_ROOT, "train")
val = load_split(DATASET_ROOT, "val")
assert set(train.speaker_id).isdisjoint(val.speaker_id), "kebocoran penutur antara latih dan validasi"
assert set(train.speaker_id) == set(meta["subsets"]["train"]["speakers"])
assert set(val.speaker_id) == set(meta["subsets"]["val"]["speakers"])
assert (train.transcription_normalized.str.len() > 0).all() and (val.transcription_normalized.str.len() > 0).all()
for df in (train, val):
    for path in df.abs_audio_path:
        load_audio(path, cfg["sampling_rate"])  # Memastikan laju sampel sesuai, audio mono, dan tidak kosong
for name, df in (("latih", train), ("validasi", val)):
    print(f"{name}: {len(df)} ujaran, {df.duration.sum() / 60:.1f} menit, penutur {sorted(df.speaker_id.unique())}, {df.quality.value_counts().to_dict()}")
print("fingerprint", meta["manifest_fingerprint"][:16])

## D. Model dan tokenizer
Prosesor, tokenizer, dan kepala CTC asli dimuat lalu **diaudit** terhadap target ternormalisasi. Karakter yang tidak dapat di-encode tidak pernah dipetakan diam-diam ke `[UNK]`.

**Penyesuaian khusus model (terdokumentasi):** vocab asli Cahya hanya memuat `a-z` dan `|`, sedangkan target kami memuat e-acute. Audit mendeteksi hal ini dan `add_missing_chars` menambahkan e-acute sebagai token CTC baru (seluruh baris `lm_head` asli dipertahankan; hanya baris baru yang diinisialisasi ulang).

In [ ]:
model, processor = load_model_and_processor(MODEL_ID, DEVICE)
texts = pd.concat([train, val]).transcription_normalized.tolist()
audit = tokenizer_audit(processor, model, texts)
print("vocab:", " ".join(audit["vocab"]))
print("laju sampel yang diharapkan:", audit["expected_sampling_rate"], "| karakter target tak tercakup:", audit["missing_target_chars"])
assert audit["expected_sampling_rate"] == cfg["sampling_rate"]

vocab_extended = []
if audit["missing_target_chars"]:
    if not ALLOW_VOCAB_EXTENSION:
        raise RuntimeError(f"Tokenizer tidak dapat meng-encode {audit['missing_target_chars']}; kepala CTC tidak diubah di notebook ini.")
    vocab_extended = audit["missing_target_chars"]
    processor, model = add_missing_chars(processor, model, vocab_extended, os.path.join(OUTPUT_ROOT, "_vocab_tmp"))
    audit = tokenizer_audit(processor, model, texts)
    print("vocab diperluas dengan", vocab_extended, "-> ukuran", audit["vocab_size"])

assert not audit["missing_target_chars"]
assert count_unk_targets(processor, texts) == 0, "masih ada target yang mengandung [UNK]"
for text in texts[:3]:
    ids = processor.tokenizer(text).input_ids
    assert processor.tokenizer.decode(ids, group_tokens=False) == text
print("encode-decode konsisten | config.vocab_size =", model.config.vocab_size, "| ctc_loss_reduction =", model.config.ctc_loss_reduction)

## E. Persiapan data
Gelombang audio mentah 16 kHz mono (dinormalisasi oleh feature extractor; tanpa MFCC dan tanpa resampling). *Collator* melakukan padding gelombang secara dinamis per batch dan mengisi label dengan `-100` agar diabaikan oleh loss CTC.

In [ ]:
train_ds, val_ds = CTCDataset(train, processor), CTCDataset(val, processor)
collator = CTCDataCollator(processor)
batch = collator([train_ds[0], train_ds[1]])
print({k: (tuple(v.shape), str(v.dtype)) for k, v in batch.items()})
assert batch["labels"].shape[0] == 2
print("label 0 terdekode:", processor.tokenizer.decode([i for i in batch["labels"][0].tolist() if i != -100], group_tokens=False))

## F. Konfigurasi pelatihan
Nilai bawaan berasal dari `CFG` pada sel kode pembantu dan identik untuk kedua notebook fine-tuning: learning rate 1e-5, 10 epoch, batch efektif 8 (batch 1 per GPU, akumulasi gradien 8 dibagi jumlah GPU sehingga 1 GPU memakai 8 dan 2 GPU memakai 4), weight decay 0,01, warmup 10%, fp16, gradient checkpointing, feature encoder CNN dibekukan, seed 42. Checkpoint terbaik adalah yang ber-**CER** validasi terendah; paling banyak 2 checkpoint disimpan; *early stopping* dengan kesabaran 3 epoch. Ubah nilai hanya pada `CFG` dan catat alasannya.

In [ ]:
if os.path.exists(os.path.join(RUN_DIR, "validation_metrics.json")) and not OVERWRITE:
    raise FileExistsError(f"{RUN_DIR} sudah berisi hasil akhir. Setel OVERWRITE=True untuk melatih ulang, atau ganti RUN_NAME.")
args = build_training_args(os.path.join(RUN_DIR, "checkpoints"), cfg)
if cfg["training"]["freeze_feature_encoder"]:
    model.freeze_feature_encoder()
if cfg["training"]["gradient_checkpointing"]:
    model.gradient_checkpointing_enable()
trainer_kwargs = dict(model=model, args=args, data_collator=collator, compute_metrics=make_compute_metrics(processor),
                      train_dataset=train_ds, eval_dataset=val_ds,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
# Nama argumen ini berbeda antar versi Transformers
trainer_kwargs["processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"] = processor
trainer = Trainer(**trainer_kwargs)
print("parameter terlatih: %.1fM" % (sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6), "|", gpu_mem())

## G. Pelatihan
Berjalan hanya di GPU (tanpa *fallback* ke CPU). Bila muncul `CUDA out of memory`: pertahankan `per_device_*_batch_size = 1`, pastikan tidak ada sesi lain yang memakai GPU, mulai ulang sesi, dan sebagai langkah terakhir (catat di dokumentasi) naikkan `gradient_accumulation_steps` atau pangkas ujaran yang sangat panjang. Sesi yang terhenti dapat dilanjutkan: jalankan ulang semua sel dengan `RESUME_FROM_CHECKPOINT = True` (checkpoint berada di `RUN_DIR/checkpoints`; simpan versi notebook beserta outputnya agar checkpoint tidak hilang antar sesi).

In [ ]:
checkpoint = last_checkpoint(args.output_dir) if RESUME_FROM_CHECKPOINT else None
print("melanjutkan dari:", checkpoint)
trainer.train(resume_from_checkpoint=checkpoint)
print(gpu_mem())

## H. Hasil

In [ ]:
hist = history_frame(trainer)
plot_history(hist).show()
ev = hist[hist["eval_cer"].notna()]
print(ev[["epoch", "eval_loss", "eval_wer", "eval_cer"]].round(4).to_string(index=False))
print("checkpoint terpilih:", trainer.state.best_model_checkpoint, "| eval_cer terbaik:", trainer.state.best_metric)
print(f"CER validasi terbaik = {ev.eval_cer.min():.4f} | WER validasi terbaik = {ev.eval_wer.min():.4f} (minimum antar epoch; pemilihan berdasarkan CER)")

Prediksi validasi akhir memakai **model terpilih** (CER terbaik) dengan dekode CTC greedy. Angka ini hanya untuk validasi dan tidak dibandingkan dengan baseline di sini (lihat `baseline/` dari notebook 01).

In [ ]:
preds = transcribe_files(trainer.model, processor, val.abs_audio_path.tolist(), batch_size=4, device=DEVICE)
result = compute_error_rates(val.transcription_normalized.tolist(), preds, normalize=False)
metrics = {"model_id": MODEL_ID, "split": "val", "wer": result["wer"], "cer": result["cer"], "details": result,
           "best_model_checkpoint": trainer.state.best_model_checkpoint, "best_eval_cer": trainer.state.best_metric}
print(f"WER validasi {result['wer']:.4f} | CER validasi {result['cer']:.4f}")

## I. Penyimpanan artefak
Model, prosesor, konfigurasi, riwayat, prediksi, dan metrik disimpan di `RUN_DIR`.

In [ ]:
cfg_record = {"model_id": MODEL_ID, "model_revision": model_revision(MODEL_ID), "seed": SEED,
              "training": cfg["training"], "vocab_extended_with": vocab_extended,
              "tokenizer": {k: v for k, v in audit.items() if k != "missing_target_chars"},
              "normalization_version": meta["normalization_version"], "decoding": "greedy CTC",
              "split": {"manifest_fingerprint": meta["manifest_fingerprint"], "seed": meta["seed"],
                        "train_speakers": meta["subsets"]["train"]["speakers"], "val_speakers": meta["subsets"]["val"]["speakers"]},
              "n_train": len(train), "n_val": len(val), "env": env_info(), "resumed_from": checkpoint}
save_artifacts(trainer, processor, RUN_DIR, cfg_record, val, preds, metrics)
for root, _, files in os.walk(RUN_DIR):
    if "checkpoints" not in root:
        print(root.replace(RUN_DIR, RUN_NAME), files)

## J. Uji kewajaran inferensi
Model dan prosesor yang **tersimpan** dimuat ulang dari disk lalu dipakai mentranskripsi beberapa ujaran validasi (tanpa data uji).

In [ ]:
del trainer, model
torch.cuda.empty_cache()
reloaded_model = Wav2Vec2ForCTC.from_pretrained(os.path.join(RUN_DIR, "model")).to(DEVICE)
reloaded_processor = Wav2Vec2Processor.from_pretrained(os.path.join(RUN_DIR, "processor"))
sample = val.iloc[:5]
reloaded_preds = transcribe_files(reloaded_model, reloaded_processor, sample.abs_audio_path.tolist(), batch_size=1, device=DEVICE)
assert reloaded_preds == preds[:5], "model termuat ulang menghasilkan prediksi berbeda dari model di memori"
for file, ref, hyp in zip(sample.file, sample.transcription_normalized, reloaded_preds):
    print(f"{file}\n  REF: {ref}\n  HIP: {hyp}")